# 01 · Transfer learning for image classification (drill)

**Style B: blind-code drill.** Write each piece from memory, run the check, delete your code.
Reference: `../solutions/drills/01_transfer_classification.py`.

**Traces to** `notes/ioai-task-patterns.md` P3, P4 · T25-G-SPEECH (torchvision ResNet18 fine-tuning),
T26-DOUBLE (ResNet18 + exact preprocessing).

Total suggested time: **50 min**.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import models
from datasets import load_dataset
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
ds = load_dataset("blanchon/EuroSAT_RGB")
def balanced(split, per_class, seed):
    lab = np.array(split["label"]); rng = np.random.default_rng(seed)
    return split.select(np.concatenate([rng.choice(np.flatnonzero(lab == c), per_class, replace=False) for c in range(10)]))
train, val = balanced(ds["train"], 120, 0), balanced(ds["validation"], 40, 1)

## 1. Train and eval transforms ⏱ 8 min

`train_tf` and `eval_tf`: callables that map a 64×64 PIL image to a normalised `float32` tensor `(3, 112, 112)`,
using ResNet18's ImageNet mean/std. `train_tf` is random (flips / crops), and `eval_tf` is deterministic.

In [ ]:
img = train[0]["image"]
a, b = eval_tf(img), eval_tf(img)
assert a.shape == (3, 112, 112) and a.dtype == torch.float32 and torch.equal(a, b)
assert any(not torch.equal(train_tf(img), train_tf(img)) for _ in range(5)), "train_tf should be random"
raw = torch.from_numpy(np.array(img.resize((112, 112)))).permute(2, 0, 1).float() / 255
want = (raw.mean((1, 2)) - torch.tensor([0.485, 0.456, 0.406])) / torch.tensor([0.229, 0.224, 0.225])
assert torch.allclose(a.mean((1, 2)), want, atol=0.05), "normalisation looks wrong"
print("section 1 ok")

## 2. Pretrained model with a new head ⏱ 5 min

`make_model(num_classes) -> nn.Module`: ResNet18 with the default ImageNet weights and a fresh `fc`.

In [ ]:
m = make_model(10)
ref = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
assert torch.equal(m.conv1.weight, ref.conv1.weight), "backbone is not pretrained"
assert m(torch.randn(2, 3, 112, 112)).shape == (2, 10)
print("section 2 ok")

## 3. Loaders and a mixed-precision epoch ⏱ 10 min

- `train_loader`, `val_loader`: over `train` / `val` (Hugging Face datasets with an `image` PIL column and a `label`
  column), yielding `(x (B,3,112,112) float32, y (B,) int64)`, using your transforms.
- `train_epoch_amp(model, loader, optimizer, scaler) -> float` mean loss, using `torch.autocast` (float16 on CUDA)
  and `GradScaler`.

In [ ]:
xb, yb = next(iter(train_loader))
assert xb.shape[1:] == (3, 112, 112) and yb.dtype == torch.int64
m = make_model(10).to(device)
opt = torch.optim.AdamW(m.parameters(), lr=5e-4)
scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")
l1 = train_epoch_amp(m, train_loader, opt, scaler)
l2 = train_epoch_amp(m, train_loader, opt, scaler)
assert l2 < l1, (l1, l2)
print(f"section 3 ok ({l1:.3f} -> {l2:.3f})")

## 4. Fine-tune to ≥ 88 % validation accuracy ⏱ 12 min

Write `evaluate(model, loader) -> accuracy`, then keep training the model `m` from section 3 until
`evaluate(m, val_loader) ≥ 0.88`.

In [ ]:
acc = evaluate(m, val_loader)
assert not m.training and acc >= 0.88, acc
print(f"section 4 ok (val accuracy {acc:.3f})")

## 5. Test-time augmentation ⏱ 6 min

`predict_tta(model, x) -> probs (B, C)`: the mean of the **softmax** over 4 views: identity, horizontal flip,
vertical flip, both flips. Eval mode, no gradients.

In [ ]:
xb = next(iter(val_loader))[0][:8].to(device)
got = predict_tta(m, xb)
with torch.no_grad():
    m.eval()
    want = sum(m(v).softmax(1) for v in [xb, xb.flip(-1), xb.flip(-2), xb.flip(-1).flip(-2)]) / 4
assert got.shape == (8, 10) and torch.allclose(got, want, atol=1e-4) and torch.allclose(got.sum(1), torch.ones(8, device=got.device), atol=1e-4)
print("section 5 ok")